In [59]:
import pandas as pd
import warnings
import numpy as np
import datetime as dt
warnings.filterwarnings('ignore')

In [96]:
df = pd.read_excel('../data/marital.xlsx')
df.rename(columns={'region':'age'}, inplace=True)
regions = df.iloc[0:18142:193, 0].str.rstrip().tolist()
n = int(len(df)/193)
cohab = []
for i in range(n):
    row_start = 26 + i*193
    row_end = 35 + i*193
    cohab.append(df.iloc[row_start:row_end])
    cohab[i]['region'] = regions[i]



In [98]:
df = pd.read_excel('../data/marital.xlsx', sheet_name='cities')
df.rename(columns={'region':'age'}, inplace=True)
regions = df.iloc[0:len(df):65,0].str.rstrip().tolist()
cohab2 = []
for i in range(2):
    row_start = 26 + i*65
    row_end = 35 + i*65
    cohab2.append(df.iloc[row_start:row_end])
    cohab2[i]['region'] = regions[i]



In [99]:
data = pd.concat(cohab + cohab2)
data_sum = data.groupby('region')[["total", 'unregistered_est']].sum().reset_index()
data_sum['unregistered_est'] = data_sum['unregistered_est']*1.5

In [100]:
df = pd.read_csv("../data/nup_2022_by_region.csv")
ex_reg = df[['region', 'year_month', 'excess_monthly']].drop_duplicates()
ex_reg = ex_reg[pd.to_datetime(ex_reg['year_month']).dt.month.isin([9, 10])]
ex_reg['excess_monthly'] = np.where(
    pd.to_datetime(ex_reg['year_month']).dt.month == 10,
    ex_reg['excess_monthly'] * 0.7,
    ex_reg['excess_monthly']
)
ex_reg = ex_reg.groupby('region')['excess_monthly'].sum().reset_index()
ex_reg

,region,excess_monthly
0,Алтайский край,1442.772467
1,Амурская область,1566.915259
2,Архангельская область,553.583056
3,Астраханская область,269.422653
4,Белгородская область,1108.688523
...,...,...
77,Челябинская область,1343.353424
78,Чеченская Республика,-320.926072
79,Чувашская Республика - Чувашия,311.358338
80,Чукотский автономный округ,57.313198


In [ ]:
set(ex_reg["region"]).symmetric_difference(set(data['region']))

reg_name_map = {
 'Город Москва столица Российской Федерации город федерального значения':'г. Москва',
 'Город Санкт-Петербург город федерального значения':'г. Санкт-Петербург',
 'Город федерального значения Севастополь':'г. Севастополь',
 'Республика Адыгея (Адыгея)':'Республика Адыгея',
 'Республика Татарстан (Татарстан)':'Республика Татарстан',
 'Чувашская Республика - Чувашия':'Чувашская Республика'
}

ex_reg["region"] = ex_reg['region'].replace(reg_name_map)
ex_reg = ex_reg.merge(data_sum, on = 'region', how = 'left')
ex_reg['mob_est'] = ex_reg['excess_monthly']/ex_reg['unregistered_est']*ex_reg['total']
ex_reg

In [108]:
ex_reg.to_csv('../data/intermediate/drafted.csv', index=False)